# SUB005 -- Multi-Template Diverse RNA 3D Prediction

**Competition**: Stanford RNA 3D Folding Part 2  
**Metric**: TM-score (higher is better, best-of-5)  
**Lineage**: SUB004 (0.211) -> SUB005

Key changes from SUB004:
- Use 5 different templates for 5 prediction slots (was 1 template + perturbations)
- Similarity-weighted template blending for one slot
- Kabsch superposition for template blending
- Local TM-score validation on validation set
- Relaxed length filtering (±50%)
- Better gap interpolation with RNA helical geometry

In [ ]:
import os
import sys
import time
import zlib
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

print("=" * 60)
print("SUB005: Multi-Template Diverse RNA 3D Prediction")
print("=" * 60)

INPUT_BASE = "/kaggle/input"
if os.path.isdir(INPUT_BASE):
    print(f"\nContents of {INPUT_BASE}:")
    for item in sorted(os.listdir(INPUT_BASE)):
        full = os.path.join(INPUT_BASE, item)
        if os.path.isdir(full):
            sub_items = os.listdir(full)
            print(f"  {item}/ ({len(sub_items)} items)")
            for si in sorted(sub_items)[:10]:
                print(f"    {si}")
        else:
            print(f"  {item} ({os.path.getsize(full)} bytes)")

In [ ]:
# ============================================================
# Data Loading
# ============================================================
CANDIDATE_BASES = [
    "/kaggle/input/stanford-rna-3d-folding-2",
    "/kaggle/input/stanford-rna-3d-folding-part-2",
    "/kaggle/input/competitions/stanford-rna-3d-folding-2",
]

DATA_PATH = None
for p in CANDIDATE_BASES:
    if os.path.isdir(p) and os.path.isfile(os.path.join(p, "test_sequences.csv")):
        DATA_PATH = p
        break

if DATA_PATH is None and os.path.isdir(INPUT_BASE):
    for root, dirs, files in os.walk(INPUT_BASE):
        if "test_sequences.csv" in files:
            DATA_PATH = root
            break

if DATA_PATH is None:
    raise FileNotFoundError(
        f"Could not find competition data. "
        f"Searched: {CANDIDATE_BASES}. "
        f"Available: {os.listdir(INPUT_BASE) if os.path.isdir(INPUT_BASE) else 'N/A'}"
    )

print(f"\nUsing data path: {DATA_PATH}")
WORK = "/kaggle/working"
os.makedirs(WORK, exist_ok=True)

train_seqs = pd.read_csv(os.path.join(DATA_PATH, "train_sequences.csv"))
test_seqs = pd.read_csv(os.path.join(DATA_PATH, "test_sequences.csv"))
train_labels = pd.read_csv(os.path.join(DATA_PATH, "train_labels.csv"))
sample_sub = pd.read_csv(os.path.join(DATA_PATH, "sample_submission.csv"))

val_seq_path = os.path.join(DATA_PATH, "validation_sequences.csv")
val_lab_path = os.path.join(DATA_PATH, "validation_labels.csv")
HAS_VAL = os.path.isfile(val_seq_path) and os.path.isfile(val_lab_path)
if HAS_VAL:
    val_seqs = pd.read_csv(val_seq_path)
    val_labels = pd.read_csv(val_lab_path)
    print(f"Validation set loaded: {len(val_seqs)} targets")
else:
    print("No validation set found")

print(f"Train sequences: {len(train_seqs)}")
print(f"Test sequences:  {len(test_seqs)}")
print(f"Train labels:    {len(train_labels)}")
print(f"Sample sub:      {len(sample_sub)}")

In [ ]:
# ============================================================
# Build template library from training data
# ============================================================
SENTINEL = -1e18

def process_labels(labels_df):
    coords_dict = {}
    prefixes = labels_df["ID"].str.rsplit("_", n=1).str[0]
    for id_prefix, group in labels_df.groupby(prefixes, sort=False):
        sorted_group = group.sort_values("resid")
        c = sorted_group[["x_1", "y_1", "z_1"]].values.astype(np.float64)
        mask = np.all(np.abs(c - SENTINEL) < 1e10, axis=1)
        if mask.any():
            c[mask] = np.nan
        coords_dict[id_prefix] = c
    return coords_dict

print("Processing training labels...")
t0 = time.time()
train_coords_dict = process_labels(train_labels)
print(f"Processed {len(train_coords_dict)} targets in {time.time()-t0:.1f}s")

TRAIN_IDS, TRAIN_SEQS, TRAIN_COORDS, TRAIN_LENS = [], [], [], []

for r in train_seqs.itertuples(index=False):
    tid = r.target_id
    if tid not in train_coords_dict:
        continue
    seq = r.sequence
    coords = train_coords_dict[tid]
    if len(coords) != len(seq):
        continue
    valid_mask = ~np.isnan(coords[:, 0])
    n_valid = valid_mask.sum()
    if n_valid < 5:
        continue
    if np.all(coords[valid_mask] == 0):
        continue
    # Store with validity info
    TRAIN_IDS.append(tid)
    TRAIN_SEQS.append(seq)
    TRAIN_COORDS.append(coords)
    TRAIN_LENS.append(len(seq))

TRAIN_LENS = np.array(TRAIN_LENS, dtype=np.int32)
print(f"Template library: {len(TRAIN_IDS)} templates with valid coordinates")
print(f"  Length range: {TRAIN_LENS.min()}-{TRAIN_LENS.max()}, mean={TRAIN_LENS.mean():.0f}")

In [ ]:
# ============================================================
# K-mer indexing for fast template retrieval
# ============================================================
KMER_K = 5
PREFILTER_TOP = 300
ALIGN_TOP = 30

_BASE2 = {"A": 0, "C": 1, "G": 2, "U": 3}

def kmer_set_2bit(seq, k=KMER_K):
    if len(seq) < k:
        return frozenset()
    mask = (1 << (2 * k)) - 1
    code = 0
    out = set()
    valid = True
    for i in range(k):
        b = _BASE2.get(seq[i])
        if b is None:
            valid = False
            break
        code = ((code << 2) | b) & mask
    if valid:
        out.add(code)
        for i in range(k, len(seq)):
            b = _BASE2.get(seq[i])
            if b is None:
                continue
            code = ((code << 2) | b) & mask
            out.add(code)
    return frozenset(out)

print("Building k-mer index...")
t0 = time.time()
TRAIN_KMERS = [kmer_set_2bit(s, KMER_K) for s in TRAIN_SEQS]
print(f"K-mer index built in {time.time()-t0:.1f}s")

In [ ]:
# ============================================================
# Alignment Functions (vectorized RNA-aware scoring)
# ============================================================
_MATCH = 2.0
_MISMATCH_TRANSITION = -0.5
_MISMATCH_TRANSVERSION = -1.5
_GAP = -2.0

_BASE_IDX = {'A': 0, 'C': 1, 'G': 2, 'U': 3}

_SCORE_MAT = np.full((5, 5), _MISMATCH_TRANSVERSION, dtype=np.float32)
for _b in range(4):
    _SCORE_MAT[_b, _b] = _MATCH
_SCORE_MAT[0, 2] = _SCORE_MAT[2, 0] = _MISMATCH_TRANSITION  # A<->G
_SCORE_MAT[1, 3] = _SCORE_MAT[3, 1] = _MISMATCH_TRANSITION  # C<->U

def _seq_to_idx(seq):
    return np.array([_BASE_IDX.get(c, 4) for c in seq], dtype=np.int8)


def needleman_wunsch(seq_a, seq_b):
    n, m = len(seq_a), len(seq_b)
    if n * m > 25_000_000:
        return _banded_nw(seq_a, seq_b, band_width=150)
    
    dp = np.zeros((n + 1, m + 1), dtype=np.float32)
    dp[0, :] = np.arange(m + 1) * _GAP
    dp[:, 0] = np.arange(n + 1) * _GAP
    
    a_idx = _seq_to_idx(seq_a)
    b_idx = _seq_to_idx(seq_b)
    
    for i in range(1, n + 1):
        scores = _SCORE_MAT[a_idx[i - 1], b_idx]
        diag = dp[i - 1, :-1] + scores
        up = dp[i - 1, 1:] + _GAP
        dp[i, 1:] = np.maximum(diag, up)
        for j in range(1, m + 1):
            dp[i, j] = max(dp[i, j], dp[i, j - 1] + _GAP)
    
    map_a_to_b = {}
    i, j = n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0:
            s = float(_SCORE_MAT[a_idx[i-1], b_idx[j-1]])
            if abs(dp[i, j] - (dp[i-1, j-1] + s)) < 1e-4:
                map_a_to_b[i - 1] = j - 1
                i -= 1
                j -= 1
                continue
        if i > 0 and abs(dp[i, j] - (dp[i-1, j] + _GAP)) < 1e-4:
            i -= 1
        else:
            j -= 1
    
    return map_a_to_b, float(dp[n, m])


def _banded_nw(seq_a, seq_b, band_width=150):
    n, m = len(seq_a), len(seq_b)
    NEGINF = -999999.0
    a_idx = _seq_to_idx(seq_a)
    b_idx = _seq_to_idx(seq_b)
    
    dp = {}
    dp[(0, 0)] = 0.0
    for j in range(1, min(band_width + 1, m + 1)):
        dp[(0, j)] = j * _GAP
    for i in range(1, min(band_width + 1, n + 1)):
        dp[(i, 0)] = i * _GAP
    
    ratio = m / n if n > 0 else 1.0
    
    for i in range(1, n + 1):
        center_j = int(i * ratio)
        j_lo = max(1, center_j - band_width)
        j_hi = min(m, center_j + band_width)
        ai = a_idx[i-1]
        for j in range(j_lo, j_hi + 1):
            s = float(_SCORE_MAT[ai, b_idx[j-1]])
            d = dp.get((i-1, j-1), NEGINF) + s
            u = dp.get((i-1, j), NEGINF) + _GAP
            l = dp.get((i, j-1), NEGINF) + _GAP
            dp[(i, j)] = max(d, u, l)
    
    score = dp.get((n, m), NEGINF)
    
    map_a_to_b = {}
    i, j = n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0:
            s = float(_SCORE_MAT[a_idx[i-1], b_idx[j-1]])
            if abs(dp.get((i, j), NEGINF) - (dp.get((i-1, j-1), NEGINF) + s)) < 1e-4:
                map_a_to_b[i - 1] = j - 1
                i -= 1
                j -= 1
                continue
        if i > 0 and abs(dp.get((i, j), NEGINF) - (dp.get((i-1, j), NEGINF) + _GAP)) < 1e-4:
            i -= 1
        elif j > 0:
            j -= 1
        else:
            break
    
    return map_a_to_b, float(score)


def nw_score_only(seq_a, seq_b):
    n, m = len(seq_a), len(seq_b)
    if n * m > 50_000_000:
        return _kmer_score_proxy(seq_a, seq_b)
    
    a_idx = _seq_to_idx(seq_a)
    b_idx = _seq_to_idx(seq_b)
    
    prev = np.arange(m + 1, dtype=np.float32) * _GAP
    
    for i in range(1, n + 1):
        curr = np.empty(m + 1, dtype=np.float32)
        curr[0] = i * _GAP
        scores = _SCORE_MAT[a_idx[i - 1], b_idx]
        diag = prev[:-1] + scores
        up = prev[1:] + _GAP
        curr[1:] = np.maximum(diag, up)
        for j in range(1, m + 1):
            curr[j] = max(curr[j], curr[j - 1] + _GAP)
        prev = curr
    
    return float(prev[m])


def _kmer_score_proxy(seq_a, seq_b, k=5):
    ka = kmer_set_2bit(seq_a, k)
    kb = kmer_set_2bit(seq_b, k)
    if not ka or not kb:
        return 0.0
    inter = len(ka & kb)
    union = len(ka | kb)
    jaccard = inter / union if union > 0 else 0.0
    return jaccard * 2.0 * min(len(seq_a), len(seq_b))

print("Alignment functions loaded.")

In [ ]:
# ============================================================
# Kabsch Superposition and Coordinate Transfer
# ============================================================
def kabsch_superpose(P, Q):
    """Kabsch algorithm: find optimal rotation+translation to align P onto Q.
    P, Q are (N, 3) arrays of matched points.
    Returns rotated P, rotation matrix R, translation t.
    """
    assert P.shape == Q.shape and P.shape[1] == 3
    centroid_P = P.mean(axis=0)
    centroid_Q = Q.mean(axis=0)
    P_c = P - centroid_P
    Q_c = Q - centroid_Q
    H = P_c.T @ Q_c
    U, S, Vt = np.linalg.svd(H)
    d = np.linalg.det(Vt.T @ U.T)
    sign_matrix = np.diag([1.0, 1.0, np.sign(d)])
    R = Vt.T @ sign_matrix @ U.T
    t = centroid_Q - R @ centroid_P
    return R, t


def transfer_coordinates(query_seq, template_seq, template_coords, alignment_map):
    qlen = len(query_seq)
    result = np.full((qlen, 3), np.nan, dtype=np.float64)
    
    for q_pos, t_pos in alignment_map.items():
        if q_pos < qlen and t_pos < len(template_coords):
            c = template_coords[t_pos]
            if not np.any(np.isnan(c)):
                result[q_pos] = c
    
    # Helical interpolation for gaps
    mapped = sorted([k for k in range(qlen) if not np.isnan(result[k, 0])])
    if not mapped:
        return _helical_chain(qlen)
    
    for i in range(qlen):
        if not np.isnan(result[i, 0]):
            continue
        prev_v = next((j for j in range(i - 1, -1, -1) if not np.isnan(result[j, 0])), -1)
        next_v = next((j for j in range(i + 1, qlen) if not np.isnan(result[j, 0])), -1)
        if prev_v >= 0 and next_v >= 0:
            gap_len = next_v - prev_v
            pos_in_gap = i - prev_v
            w = pos_in_gap / gap_len
            base = (1 - w) * result[prev_v] + w * result[next_v]
            if gap_len > 2:
                # Add helical deviation for multi-residue gaps
                direction = result[next_v] - result[prev_v]
                norm_d = np.linalg.norm(direction)
                if norm_d > 1e-6:
                    perp = np.cross(direction, [0, 0, 1])
                    perp_norm = np.linalg.norm(perp)
                    if perp_norm < 1e-6:
                        perp = np.cross(direction, [0, 1, 0])
                        perp_norm = np.linalg.norm(perp)
                    perp = perp / (perp_norm + 1e-12)
                    perp2 = np.cross(direction / norm_d, perp)
                    angle = 2 * np.pi * pos_in_gap / max(gap_len, 1)
                    # RNA A-form radius ~9.4A but scaled by gap size
                    radius = min(2.0, 0.5 * gap_len)
                    helix_offset = radius * (np.cos(angle) * perp + np.sin(angle) * perp2)
                    # Dampen at endpoints
                    dampen = np.sin(np.pi * w)
                    base += helix_offset * dampen
            result[i] = base
        elif prev_v >= 0:
            # Extend with RNA-like spacing along last direction
            if prev_v > 0 and not np.isnan(result[prev_v - 1, 0]):
                d = result[prev_v] - result[prev_v - 1]
                d = d / (np.linalg.norm(d) + 1e-12) * 5.95
            else:
                d = np.array([5.95, 0, 0])
            result[i] = result[prev_v] + d * (i - prev_v)
        elif next_v >= 0:
            if next_v < qlen - 1 and not np.isnan(result[next_v + 1, 0]):
                d = result[next_v + 1] - result[next_v]
                d = d / (np.linalg.norm(d) + 1e-12) * 5.95
            else:
                d = np.array([5.95, 0, 0])
            result[i] = result[next_v] - d * (next_v - i)
        else:
            result[i] = [i * 5.95, 0, 0]
    
    return np.nan_to_num(result)


def _helical_chain(length, spacing=5.95, twist_deg=32.7, radius=2.0):
    coords = np.zeros((length, 3), dtype=np.float64)
    twist_rad = np.deg2rad(twist_deg)
    rise = spacing * 0.47  # A-form helix rise per residue
    for i in range(length):
        angle = i * twist_rad
        coords[i, 0] = radius * np.cos(angle)
        coords[i, 1] = radius * np.sin(angle)
        coords[i, 2] = i * rise
    return coords


def blend_templates(query_seq, templates, weights):
    """Blend multiple template coordinate transfers using Kabsch superposition.
    templates: list of (template_seq, template_coords, alignment_map, similarity)
    weights: array of blend weights
    """
    qlen = len(query_seq)
    if not templates:
        return _helical_chain(qlen)
    
    # Get individual transfers
    transfers = []
    for t_seq, t_coords, amap, sim in templates:
        coords = transfer_coordinates(query_seq, t_seq, t_coords, amap)
        transfers.append(coords)
    
    if len(transfers) == 1:
        return transfers[0]
    
    # Use first transfer as reference frame
    ref = transfers[0]
    aligned = [ref]
    
    for k in range(1, len(transfers)):
        other = transfers[k]
        # Find shared valid positions
        valid = ~(np.isnan(ref[:, 0]) | np.isnan(other[:, 0]))
        n_valid = valid.sum()
        if n_valid >= 4:
            R, t = kabsch_superpose(other[valid], ref[valid])
            other_aligned = (other @ R.T) + t
            aligned.append(other_aligned)
        else:
            aligned.append(other)
    
    # Weighted average
    w = np.array(weights[:len(aligned)], dtype=np.float64)
    w = w / (w.sum() + 1e-12)
    result = np.zeros((qlen, 3), dtype=np.float64)
    for k in range(len(aligned)):
        result += w[k] * aligned[k]
    
    return result

print("Coordinate transfer and blending loaded.")

In [ ]:
# ============================================================
# Template Retrieval (improved)
# ============================================================
def find_similar_sequences(query_seq, top_n=ALIGN_TOP):
    qlen = len(query_seq)
    qkm = kmer_set_2bit(query_seq, KMER_K)
    qkm_len = len(qkm)
    
    if len(TRAIN_IDS) == 0:
        return []
    
    # Relaxed length filter: ±50%
    maxL = np.maximum(TRAIN_LENS, qlen)
    keep = (np.abs(TRAIN_LENS - qlen) / maxL) <= 0.50
    idxs = np.where(keep)[0]
    if idxs.size == 0:
        keep = (np.abs(TRAIN_LENS - qlen) / maxL) <= 0.70
        idxs = np.where(keep)[0]
    if idxs.size == 0:
        # Take closest by length
        diffs = np.abs(TRAIN_LENS - qlen)
        idxs = np.argsort(diffs)[:PREFILTER_TOP]
    
    scored = []
    for i in idxs:
        tkm = TRAIN_KMERS[i]
        if qkm_len == 0 or len(tkm) == 0:
            jac = 0.0
        else:
            inter = len(qkm & tkm)
            union = qkm_len + len(tkm) - inter
            jac = inter / union if union else 0.0
        scored.append((jac, int(i)))
    
    scored.sort(key=lambda x: x[0], reverse=True)
    top_candidates = scored[:PREFILTER_TOP]
    
    NW_CELL_LIMIT = 5_000_000
    sims = []
    for jac, i in top_candidates:
        t_seq = TRAIN_SEQS[i]
        if qlen * len(t_seq) < NW_CELL_LIMIT:
            raw = nw_score_only(query_seq, t_seq)
            denom = 2.0 * min(qlen, len(t_seq))
            norm = float(raw / denom) if denom > 0 else jac
        else:
            norm = jac
        sims.append((TRAIN_IDS[i], t_seq, norm, TRAIN_COORDS[i], i))
    
    sims.sort(key=lambda x: x[2], reverse=True)
    return sims[:top_n]

print("Template retrieval ready.")

In [ ]:
# ============================================================
# Stoichiometry and Chain Handling
# ============================================================
def parse_fasta(fasta_content):
    out = {}
    cur = None
    seq_parts = []
    for line in str(fasta_content).splitlines():
        line = line.strip()
        if not line:
            continue
        if line.startswith(">"):
            if cur is not None:
                out[cur] = "".join(seq_parts)
            cur = line[1:].split()[0]
            seq_parts = []
        else:
            seq_parts.append(line.replace(" ", ""))
    if cur is not None:
        out[cur] = "".join(seq_parts)
    return out


def parse_stoichiometry(stoich):
    if pd.isna(stoich) or str(stoich).strip() == "":
        return []
    out = []
    for part in str(stoich).split(";"):
        parts = part.split(":")
        if len(parts) == 2:
            out.append((parts[0].strip(), int(parts[1])))
    return out


def get_chain_segments(row):
    seq = row["sequence"]
    stoich = row.get("stoichiometry", "")
    all_seq = row.get("all_sequences", "")
    
    if pd.isna(stoich) or pd.isna(all_seq) or str(stoich).strip() == "" or str(all_seq).strip() == "":
        return [(0, len(seq))]
    
    try:
        chain_dict = parse_fasta(all_seq)
        order = parse_stoichiometry(stoich)
        segs = []
        pos = 0
        for ch, cnt in order:
            base = chain_dict.get(ch)
            if base is None:
                return [(0, len(seq))]
            for _ in range(cnt):
                L = len(base)
                segs.append((pos, pos + L))
                pos += L
        if pos != len(seq):
            return [(0, len(seq))]
        return segs
    except Exception:
        return [(0, len(seq))]


test_segs_map = {}
for _, r in test_seqs.iterrows():
    test_segs_map[r["target_id"]] = get_chain_segments(r)

print(f"Chain segments computed for {len(test_segs_map)} test targets")
for tid, segs in list(test_segs_map.items())[:3]:
    total_res = sum(e-s for s,e in segs)
    print(f"  {tid}: {len(segs)} chain(s), total {total_res} residues")

In [ ]:
# ============================================================
# RNA Structural Constraints
# ============================================================
def adaptive_rna_constraints(coordinates, segments, confidence=1.0, passes=3):
    coords = coordinates.copy()
    strength = 0.75 * (1.0 - min(confidence, 0.90))
    strength = max(strength, 0.03)
    
    for _ in range(passes):
        for (s, e) in segments:
            X = coords[s:e]
            L = e - s
            if L < 3:
                continue
            
            # C1'-C1' bond distance (~5.95 Angstrom)
            d = X[1:] - X[:-1]
            dist = np.linalg.norm(d, axis=1) + 1e-6
            target = 5.95
            scale = (target - dist) / dist
            adj = (d * scale[:, None]) * (0.25 * strength)
            X[:-1] -= adj
            X[1:] += adj
            
            # Second-neighbor distance
            if L >= 5:
                d2 = X[2:] - X[:-2]
                dist2 = np.linalg.norm(d2, axis=1) + 1e-6
                target2 = 10.2
                scale2 = (target2 - dist2) / dist2
                adj2 = (d2 * scale2[:, None]) * (0.12 * strength)
                X[:-2] -= adj2
                X[2:] += adj2
            
            # Laplacian smoothing
            if L >= 5:
                lap = 0.5 * (X[:-2] + X[2:]) - X[1:-1]
                X[1:-1] += (0.08 * strength) * lap
            
            # Steric repulsion
            if L >= 25:
                k = min(L, 200)
                idx = np.linspace(0, L - 1, k).astype(int) if k < L else np.arange(L)
                P = X[idx]
                diff = P[:, None, :] - P[None, :, :]
                distm = np.linalg.norm(diff, axis=2) + 1e-6
                sep = np.abs(idx[:, None] - idx[None, :])
                mask = (sep > 2) & (distm < 3.2)
                if np.any(mask):
                    force = (3.2 - distm) / distm
                    vec = (diff * force[:, :, None] * mask[:, :, None]).sum(axis=1)
                    X[idx] += (0.02 * strength) * vec
            
            coords[s:e] = X
    
    return coords

print("Structural constraints loaded.")

In [ ]:
# ============================================================
# TM-score Computation
# ============================================================
def compute_tm_score(pred_coords, true_coords):
    """Compute TM-score between predicted and true C1' coordinates.
    Uses Kabsch superposition for optimal alignment.
    """
    valid = ~(np.isnan(true_coords[:, 0]) | np.isnan(pred_coords[:, 0]))
    if valid.sum() < 3:
        return 0.0
    
    pred = pred_coords[valid]
    true = true_coords[valid]
    L = len(true)
    
    # d0 normalization factor
    if L < 15:
        d0 = 0.5
    else:
        d0 = 0.6 * np.sqrt(L - 0.5) - 2.5
        d0 = max(d0, 0.5)
    
    # Kabsch alignment
    try:
        R, t = kabsch_superpose(pred, true)
        pred_aligned = (pred @ R.T) + t
    except Exception:
        pred_aligned = pred
    
    # TM-score
    distances = np.linalg.norm(pred_aligned - true, axis=1)
    tm = np.sum(1.0 / (1.0 + (distances / d0) ** 2)) / L
    
    return float(tm)


def compute_best_of_5_tm(all_preds, true_coords):
    """Compute best TM-score among 5 predictions."""
    best = 0.0
    for pred in all_preds:
        tm = compute_tm_score(pred, true_coords)
        best = max(best, tm)
    return best

print("TM-score computation loaded.")

In [ ]:
# ============================================================
# Multi-Template Prediction Pipeline
# ============================================================
def predict_rna_structures(tid, seq, segments, n_predictions=5):
    """Predict n diverse 3D structures using different templates per slot."""
    cands = find_similar_sequences(query_seq=seq, top_n=ALIGN_TOP)
    predictions = []
    
    if not cands:
        for _ in range(n_predictions):
            predictions.append(_helical_chain(len(seq)))
        return predictions
    
    # Pre-compute alignments for top candidates
    aligned_templates = []
    for c_id, c_seq, c_sim, c_coords, c_idx in cands[:min(10, len(cands))]:
        amap, score = needleman_wunsch(seq, c_seq)
        aligned_templates.append((c_id, c_seq, c_coords, amap, c_sim, score))
    
    for i in range(n_predictions):
        if i == 0:
            # Slot 1: Best template, direct transfer
            _, t_seq, t_coords, amap, sim, _ = aligned_templates[0]
            X = transfer_coordinates(seq, t_seq, t_coords, amap)
        
        elif i == 1 and len(aligned_templates) >= 2:
            # Slot 2: Second-best template
            _, t_seq, t_coords, amap, sim, _ = aligned_templates[1]
            X = transfer_coordinates(seq, t_seq, t_coords, amap)
        
        elif i == 2 and len(aligned_templates) >= 3:
            # Slot 3: Weighted blend of top-3 templates
            blend_input = []
            blend_weights = []
            for j in range(min(3, len(aligned_templates))):
                _, t_seq, t_coords, amap, sim, _ = aligned_templates[j]
                blend_input.append((t_seq, t_coords, amap, sim))
                blend_weights.append(max(sim, 0.01))
            X = blend_templates(seq, blend_input, blend_weights)
            sim = aligned_templates[0][4]
        
        elif i == 3 and len(aligned_templates) >= 4:
            # Slot 4: Fourth-best template
            _, t_seq, t_coords, amap, sim, _ = aligned_templates[3]
            X = transfer_coordinates(seq, t_seq, t_coords, amap)
        
        elif i == 4 and len(aligned_templates) >= 5:
            # Slot 5: Fifth-best template  
            _, t_seq, t_coords, amap, sim, _ = aligned_templates[4]
            X = transfer_coordinates(seq, t_seq, t_coords, amap)
        
        else:
            # Fallback: use best available with small perturbation
            idx = min(i, len(aligned_templates) - 1)
            _, t_seq, t_coords, amap, sim, _ = aligned_templates[idx]
            X = transfer_coordinates(seq, t_seq, t_coords, amap)
            seed = (zlib.adler32(f"{tid}_{i}".encode()) & 0xFFFFFFFF)
            rng = np.random.default_rng(seed)
            noise_std = max(0.3, (0.5 - sim) * 2.0)
            X += rng.normal(0, noise_std, X.shape)
        
        # Apply structural constraints
        sim_val = aligned_templates[min(i, len(aligned_templates)-1)][4]
        refined = adaptive_rna_constraints(X, segments, confidence=sim_val, passes=3)
        predictions.append(refined)
    
    return predictions

print("Multi-template prediction pipeline ready.")

In [ ]:
# ============================================================
# Validation (if available)
# ============================================================
if HAS_VAL:
    print("\n" + "=" * 60)
    print("Running validation (max 20 targets)...")
    print("=" * 60)
    
    val_segs_map = {}
    for _, r in val_seqs.iterrows():
        val_segs_map[r["target_id"]] = get_chain_segments(r)
    
    val_coords_dict = process_labels(val_labels)
    
    val_tm_scores = []
    t0 = time.time()
    val_count = 0
    for _, r in val_seqs.iterrows():
        if val_count >= 20:
            break
        tid = r["target_id"]
        seq = r["sequence"]
        if len(seq) > 2000:
            continue
        segs = val_segs_map.get(tid, [(0, len(seq))])
        
        if tid not in val_coords_dict:
            continue
        true_coords = val_coords_dict[tid]
        if len(true_coords) != len(seq):
            continue
        
        preds = predict_rna_structures(tid, seq, segs, n_predictions=5)
        tm = compute_best_of_5_tm(preds, true_coords)
        val_tm_scores.append((tid, tm, len(seq)))
        val_count += 1
    
    val_time = time.time() - t0
    
    if val_tm_scores:
        scores = [s[1] for s in val_tm_scores]
        print(f"\nValidation results ({len(val_tm_scores)} targets, {val_time:.1f}s):")
        print(f"  Mean TM-score: {np.mean(scores):.4f}")
        print(f"  Median TM-score: {np.median(scores):.4f}")
        print(f"  Min: {np.min(scores):.4f}, Max: {np.max(scores):.4f}")
        print(f"  Scores > 0.3: {sum(1 for s in scores if s > 0.3)}/{len(scores)}")
        print(f"  Scores > 0.4: {sum(1 for s in scores if s > 0.4)}/{len(scores)}")
        print(f"\n  Per-target (top 10 by score):")
        for tid, tm, L in sorted(val_tm_scores, key=lambda x: -x[1])[:10]:
            print(f"    {tid}: TM={tm:.4f} (L={L})")
        print(f"\n  Per-target (bottom 5 by score):")
        for tid, tm, L in sorted(val_tm_scores, key=lambda x: x[1])[:5]:
            print(f"    {tid}: TM={tm:.4f} (L={L})")
    else:
        print("No valid validation targets found")
else:
    print("Skipping validation (no validation set available)")

In [ ]:
# ============================================================
# Generate Predictions for All Test Targets
# ============================================================
print(f"\nGenerating predictions for {len(test_seqs)} test targets...")
start_time = time.time()

dfs = []

for idx, r in enumerate(test_seqs.itertuples(index=False)):
    tid = r.target_id
    seq = r.sequence
    L = len(seq)
    segments = test_segs_map.get(tid, [(0, L)])
    
    t0 = time.time()
    preds = predict_rna_structures(tid, seq, segments, n_predictions=5)
    elapsed = time.time() - t0
    
    if idx < 5 or idx % 10 == 0:
        print(f"  [{idx+1}/{len(test_seqs)}] {tid} (L={L}) -> {elapsed:.1f}s")
    
    data = {
        "ID": [f"{tid}_{j}" for j in range(1, L + 1)],
        "resname": list(seq),
        "resid": np.arange(1, L + 1, dtype=np.int32),
    }
    for i in range(5):
        data[f"x_{i+1}"] = preds[i][:, 0].astype(np.float32)
        data[f"y_{i+1}"] = preds[i][:, 1].astype(np.float32)
        data[f"z_{i+1}"] = preds[i][:, 2].astype(np.float32)
    
    dfs.append(pd.DataFrame(data))

total_time = time.time() - start_time
print(f"\nAll predictions generated in {total_time:.1f}s")

In [ ]:
# ============================================================
# Build and Save Submission
# ============================================================
sub = pd.concat(dfs, ignore_index=True)

cols = ["ID", "resname", "resid"] + [f"{c}_{i}" for i in range(1, 6) for c in ["x", "y", "z"]]
coord_cols = [c for c in cols if c.startswith(("x_", "y_", "z_"))]

sub[coord_cols] = sub[coord_cols].clip(-999.999, 9999.999)
sub = sub.fillna(0.0)

output_path = os.path.join(WORK, "submission.csv")
sub[cols].to_csv(output_path, index=False)

print(f"Submission saved to {output_path}")
print(f"Shape: {sub.shape}")
print(f"Expected: {len(sample_sub)} rows")
print(f"NaN values: {sub[coord_cols].isna().sum().sum()}")
print(f"Zero values: {(sub[coord_cols] == 0).sum().sum()}")
print()
print(sub.head())
print()
print("=" * 60)
print("SUB005 Pipeline Complete")
print("=" * 60)
print(f"  Templates used  : {len(TRAIN_IDS)}")
print(f"  Test targets    : {len(test_seqs)}")
print(f"  Submission rows : {len(sub)}")
print(f"  Total time      : {total_time:.1f}s")
print("=" * 60)